<a href="https://colab.research.google.com/github/kiran162005/flyrank-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kiran162005/flyrank-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### Feature Vector

For my Refresh / Content Opportunity Scoring lane, the unit of analysis is one webpage at a decision moment.

I will use observable search and content-performance signals that are available before the refresh decision. The feature vector will contain a small number of numeric and categorical features rather than every available field.

The initial features are:

* `content_age_days` — age of the webpage.
* `days_since_last_update` — how long it has been since the page was updated.
* `impressions_90d` — search visibility/exposure over the available 90-day window.
* `avg_position` — average search position.
* `ctr` — click-through rate.

Missing numeric values will be filled with 0 for this starter feature vector. Categorical encoding is not required for these five initial numeric features.

The purpose is to create a compact, interpretable feature vector that can support ranking webpages for content-review priority.


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("https://raw.githubusercontent.com/kiran162005/flyrank-ml/main/data/raw/content_refresh_anonymized.csv")

feature_cols = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr"
]

X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

print("Feature vector shape:", X.shape)
display(X.head())

Feature vector shape: (30000, 5)


,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr
0,187,20,3803,10.6,0.76
1,445,25,15320,20.3,0.05
2,141,20,12581,36.5,0.09
3,463,22,11751,6.2,0.49
4,263,14,19140,44.0,0.13


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature Notes

| Feature                  | Meaning                                             | Missing values | Categorical? | Available when?                                                                                |
| ------------------------ | --------------------------------------------------- | -------------- | ------------ | ---------------------------------------------------------------------------------------------- |
| `content_age_days`       | Age of the webpage in days                          | Filled with 0  | No           | Knowable at the decision moment because the page's age is already observable.                  |
| `days_since_last_update` | Days since the page was last updated                | Filled with 0  | No           | Knowable at the decision moment because the update history already exists.                     |
| `impressions_90d`        | Search impressions over the available 90-day window | Filled with 0  | No           | Knowable at the decision moment because it summarizes previously observed search exposure.     |
| `avg_position`           | Average search position                             | Filled with 0  | No           | Knowable at the decision moment because it is calculated from observed search performance.     |
| `ctr`                    | Click-through rate                                  | Filled with 0  | No           | Knowable at the decision moment because it is calculated from observed impressions and clicks. |

These features are intended to represent information available before a refresh recommendation is acted on. They are used for decision-support rather than causal claims.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage Hunt

I will deliberately test a label-derived feature to demonstrate leakage.

The outcome label for this lane is based on whether `trend_direction` is `"down"`. Therefore, `trend_direction` and `trend_pct` contain information derived from the outcome and should not be used as predictive features.

If I include `trend_pct`, the model may obtain an unrealistically high score because it is seeing information that is effectively the answer.

I will compare a model using the honest features with a model that includes the deliberately leaked feature. After the test, the leaked feature will be removed from the final feature vector.


In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

df["is_declining_label"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

y = df["is_declining_label"]

print("Declining rate:", round(y.mean(), 3))

Declining rate: 0.542


In [4]:
honest_features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr"
]

X_honest = (
    df[honest_features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

honest_model = DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=42
)

honest_model.fit(X_honest, y)

honest_score = honest_model.predict_proba(X_honest)[:, 1]
honest_pred = (honest_score >= 0.5).astype(int)

print("Honest feature accuracy:",
      round(accuracy_score(y, honest_pred), 3))

Honest feature accuracy: 0.637


In [5]:
leaky_features = honest_features + ["trend_pct"]

X_leaky = (
    df[leaky_features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

leaky_model = DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=42
)

leaky_model.fit(X_leaky, y)

leaky_score = leaky_model.predict_proba(X_leaky)[:, 1]
leaky_pred = (leaky_score >= 0.5).astype(int)

print("Leaky feature accuracy:",
      round(accuracy_score(y, leaky_pred), 3))

Leaky feature accuracy: 1.0


### Leakage Conclusion

The deliberately leaked feature produced an unrealistically strong result because `trend_pct` is derived from the same underlying trend information used to define the declining label.

I therefore removed `trend_pct` from the final feature vector.

The final model features contain only observable signals that are intended to be available before the refresh decision. The leakage experiment is useful as a warning, but the leaky score is not a valid model result.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded Fields

* **`trend_direction`** — excluded because it is directly used to define the declining label.
* **`trend_pct`** — excluded because it is derived from the trend outcome and can leak the label into the model.
* **Any future-period performance fields** — excluded because they would not be available at the moment the refresh decision is made.
* **Client-identifying information** — excluded because the project only needs page-level signals for decision-support and must follow the dataset's privacy and data-use restrictions.
* **Existing decision/product flags** — excluded because they may represent decisions or classifications already made by the system rather than independent evidence available before the decision.

The final feature vector is intentionally small and interpretable. It is designed to support prioritization, not to prove that a refresh will cause better performance.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.